In [4]:
import numpy as np
from scipy.stats import norm, multivariate_normal
from scipy.optimize import minimize
from scipy.special import expit
import torch
import math

## **Gaussian Copula**

$$
\begin{array}{ccl}
\Phi(x) &:& \text{standard normal CDF} \quad \rightarrow \quad \texttt{norm.cdf(x)}
\\
\Phi^{-1}(u) &:& \text{standard normal inverse CDF}\quad \rightarrow \quad \texttt{norm.ppf(u)}
\\
\phi(x) &:& \text{standard normal density}\quad \rightarrow \quad \texttt{norm.pdf(x)}
\\
\Phi_\rho(x,y) &:& \text{bivariate standard normal CDF with correlation } \rho \quad \rightarrow \quad \texttt{multivariate\_normal.cdf([x, y], mean, cov)}
\\
\phi_\rho(x,y) &:& \text{bivariate standard normal density with correlation } \rho \quad \rightarrow \quad \texttt{multivariate\_normal.pdf([x, y], mean, cov)}
\end{array}
$$

 **Gaussian Copula Density $c_\rho$**

$$
\begin{array}{ccl}
c_\rho(u,v) &=& \displaystyle\frac{\mathcal{N}_2\{\Phi^{-1}(u),\Phi^{-1}(v)|0,1,\rho\}}{\mathcal{N}\{\Phi^{-1}(u)|0,1\}\mathcal{N}\{\Phi^{-1}(v)|0,1\}} \\
&=& \displaystyle\frac{1}{\sqrt{1-\rho^2}} \exp\left[-\displaystyle\frac{\rho^2(z_u^2+z_v^2)-2\rho z_u z_v}{2(1-\rho^2)}\right]
\end{array}
$$

where
$$
z_u = \Phi^{-1}(u),
\qquad
z_v = \Phi^{-1}(v)
$$

In [10]:
def gaussian_copula_density(u, v, rho):

    u = np.clip(u, 1e-6, 1 - 1e-6)
    v = np.clip(v, 1e-6, 1 - 1e-6)
    rho = np.clip(rho, -1 + 1e-6, 1 - 1e-6)

    z_u = norm.ppf(u)
    z_v = norm.ppf(v)

    numerator = np.exp( - (rho**2 * (z_u**2 + z_v**2) - 2 * rho * z_u * z_v) / (2 * (1 - rho**2)))
    denominator = np.sqrt(1 - rho**2)

    return numerator / denominator

 **Gaussian Copula CDF** $C_\rho$

$$
C_\rho(u,v)
= \Phi_\rho\left(\Phi^{-1}(u),\Phi^{-1}(v)\right)
$$

where
$$
z_u = \Phi^{-1}(u), 
\qquad 
z_v = \Phi^{-1}(v)
$$

In [11]:
def gaussian_copula_cdf(u, v, rho):

    u = np.clip(u, 1e-6, 1 - 1e-6)
    v = np.clip(v, 1e-6, 1 - 1e-6)
    rho = np.clip(rho, -1 + 1e-6, 1 - 1e-6)

    z_u = norm.ppf(u)
    z_v = norm.ppf(v)

    mean = [0, 0]
    cov = [[1, rho], [rho, 1]]

    return multivariate_normal.cdf([z_u, z_v], mean=mean, cov=cov)


**Conditional Gaussian Copula CDF** $H_\rho$

$$
H_\rho(u,v)
=
\Phi
\left(
\frac{
\Phi^{-1}(u)-\rho\Phi^{-1}(v)
}{
\sqrt{1-\rho^2}
}
\right)
$$

In [13]:
def gaussian_conditional_copula_cdf(u, v, rho):

    u = np.clip(u, 1e-6, 1 - 1e-6)
    v = np.clip(v, 1e-6, 1 - 1e-6)
    rho = np.clip(rho, -1 + 1e-6, 1 - 1e-6)

    z_u = norm.ppf(u)
    z_v = norm.ppf(v)

    return norm.cdf((z_u - rho * z_v) / np.sqrt(1 - rho**2))

**Multivariate Gaussian Copula Density $c_R$**

For $d$-dimensional $u=(u_1,\ldots,u_d)$,

$$
c_R(u_1,\ldots,u_d)
=
\frac{\phi_R(z_1,\ldots,z_d)}
{\prod_{j=1}^{d}\phi(z_j)}
$$

where

$$
z_j = \Phi^{-1}(u_j), \qquad j=1,\ldots,d.
$$

Equivalently,

$$
c_R(u)
=
|R|^{-1/2}
\exp\left[
-\frac{1}{2}
z^\top (R^{-1}-I)z
\right],
$$

where

$$
z=(z_1,\ldots,z_d)^\top,
\qquad
R \text{ is the correlation matrix.}
$$

In [1]:
def gaussian_copula_density_multivariate(u, R):

    eps = 1e-6

    u = np.asarray(u)
    u = np.clip(u, eps, 1 - eps)

    z = norm.ppf(u)

    d = len(z)

    R = np.asarray(R)

    sign, logdet = np.linalg.slogdet(R)

    if sign <= 0:
        return 1e-12

    R_inv_z = np.linalg.solve(R, z)

    exponent = -0.5 * (z @ R_inv_z - z @ z)

    log_density = -0.5 * logdet + exponent

    density = np.exp(log_density)

    density = max(density, 1e-12)

    return density

In [4]:
u = np.array([0.3, 0.7, 0.5])

R = np.array([
    [1.0, 0.5, 0.5],
    [0.5, 1.0, 0.5],
    [0.5, 0.5, 1.0]
])

gaussian_copula_density_multivariate(u, R)

np.float64(1.074201604922916)

## **Vine copula**

In [18]:
import pyvinecopulib as pv

$$
\text{vine}_1\quad:\quad \hat c_{vine}^{(1)}
\left(U_{t+1}^{(1)}, U_t,\ldots , U_{t-k+1}\right)
$$

$$
\text{vine}_2\quad:\quad\hat c_{vine}^{(2)}
\left(U_{t+1}^{(2)}, U_t, \ldots, U_{t-k+1}\right)
$$

$$
\vdots
$$


$$
\text{vine}_d\quad:\quad \hat c_{vine}^{(d)}
\left(U_{t+1}^{(d)}, U_t, \ldots, U_{t-k+1}\right)
$$

In [3]:
def fit_conditional_vines(U_condition, U_target):
    
    U_condition = np.asarray(U_condition)
    U_target = np.asarray(U_target)

    d = U_target.shape[1]

    vines = []
    for j in range(d):

        data_j = np.column_stack([U_target[:, j], U_condition])
        
        vine_j = pv.Vinecop.from_data(data_j)
        vines.append(vine_j)

    return vines

$$
\widehat{c}^{(j)}
\left(u \mid u_{\text{condition}}\right)
=\frac{\widehat{c}^{(j)}
\left(u, u_{\text{condition}}\right)
}{
\displaystyle\int_0^1\widehat{c}^{(j)}\left(s, u_{\text{condition}}\right)\, ds}
$$

In [1]:
def conditional_vine_copula_pdf(vine, u, u_condition, n_grid=300):

    eps = 1e-6
    u = np.clip(u, eps, 1 - eps)
    u_condition = np.asarray(u_condition)

    s_grid = np.linspace(eps, 1 - eps, n_grid)

    numerator_input = np.concatenate([[u], u_condition]).reshape(1, -1)
    numerator = vine.pdf(numerator_input)[0]

    denom_inputs = np.column_stack([s_grid, np.tile(u_condition, (n_grid, 1))])

    denom_values = vine.pdf(denom_inputs) 
    denominator = np.trapz(denom_values, s_grid) # integral

    cond_pdf = numerator / denominator
    cond_pdf = max(cond_pdf, 1e-12)

    return cond_pdf

$$
\widehat{C}^{(j)}
\left(
u \mid u_{\text{condition}}
\right)
=\frac{\displaystyle\int_0^u\widehat{c}^{(j)}\left(s, u_{\text{condition}}\right)\, ds
}{\displaystyle\int_0^1\widehat{c}^{(j)}\left(s, u_{\text{condition}}\right)\, ds}
$$

In [14]:
def conditional_vine_copula_cdf(vine, u, u_condition, n_grid=300):
    
    eps = 1e-6
    u = np.clip(u, eps, 1 - eps)
    u_condition = np.asarray(u_condition)

    # denominator
    s_full = np.linspace(eps, 1 - eps, n_grid)

    denom_inputs = np.column_stack([s_full, np.tile(u_condition, (n_grid, 1))])

    denom_values = vine.pdf(denom_inputs)
    denominator = np.trapz(denom_values, s_full)

    # numerator
    s_part = np.linspace(eps, u, n_grid)

    numer_inputs = np.column_stack([s_part, np.tile(u_condition, (n_grid, 1))])

    numer_values = vine.pdf(numer_inputs)
    numerator = np.trapz(numer_values, s_part)

    cond_cdf = numerator / denominator
    cond_cdf = np.clip(cond_cdf, 0.0, 1.0)

    return cond_cdf

## **Time-varying Copula**

### **GAS model**

$$
\widetilde{\mathbf U}_i
$$

$$
\downarrow
$$

$$
\mathbf Z_i=\Phi^{-1}\!\left(\widetilde{\mathbf U}_i\right)
$$

$$
\mathbf f_{G,i}
\qquad\qquad\qquad
f_{C,i}
$$

$$
\downarrow\qquad\qquad\qquad\downarrow
$$

$$
\texttt{gaussian\_parameter\_3d}
\!\left(\mathbf f_{G,i}\right)
\qquad
\texttt{clayton\_parameter}
\!\left(f_{C,i}\right)
$$

$$
\downarrow\qquad\qquad\qquad\downarrow
$$

$$
\mathbf R_i
\qquad\qquad\qquad
\theta_i
$$

$$
\left(\mathbf Z_i,\mathbf R_i\right)
\qquad\qquad
\left(\widetilde{\mathbf U}_i,\theta_i\right)
$$

$$
\downarrow\qquad\qquad\qquad\downarrow
$$

$$
\texttt{gaussian\_copula\_log\_density}
\!\left(\mathbf Z_i,\mathbf R_i\right)
\qquad
\texttt{clayton\_copula\_log\_density}
\!\left(\widetilde{\mathbf U}_i,\theta_i\right)
$$

$$
\downarrow\qquad\qquad\qquad\downarrow
$$

$$
\log c_{G,i}
\qquad\qquad\qquad
\log c_{C,i}
$$

$$
\downarrow
$$

$$
\texttt{gaussian\_clayton\_mixture\_log\_density}
\!\left(
\log c_{G,i},
\log c_{C,i},
\eta
\right)
$$

$$
\downarrow
$$

$$
w=\sigma(\eta),
\qquad
\log c_{\mathrm{mix},i}
=\log\!\left[
w\,c_{G,i}
+
(1-w)c_{C,i}
\right]
$$

$$
\downarrow
$$

$$
\texttt{gaussian\_clayton\_mixture\_raw\_score}
\!\left(\log c_{\mathrm{mix},i},\mathbf f_{G,i},f_{C,i}\right)
$$

$$
\downarrow
$$

$$
\mathbf r_{G,i}=\nabla_{\mathbf f_{G,i}}\log c_{\mathrm{mix},i},
\qquad
r_{C,i}=\frac{\partial\log c_{\mathrm{mix},i}}{\partial f_{C,i}}
$$

$$
\mathbf r_{G,i}=\tau_{G,i}\nabla_{\mathbf f_{G,i}}\log c_{G,i},
\qquad
r_{C,i}=\tau_{C,i}\frac{\partial\log c_{C,i}}{\partial f_{C,i}}
$$

$$
\tau_{G,i}
=\frac{w\,c_{G,i}}{w\,c_{G,i}+(1-w)c_{C,i}},
\qquad
\tau_{C,i}=\frac{(1-w)c_{C,i}}{w\,c_{G,i}+(1-w)c_{C,i}}=1-\tau_{G,i}
$$

$$
\downarrow
$$

$$
\texttt{identity\_scaling}
\!\left(
\mathbf r_{G,i},
r_{C,i}
\right)
$$

$$
\downarrow
$$

$$
\mathbf s_{G,i}=\mathbf r_{G,i},
\qquad
s_{C,i}=r_{C,i}
$$

$$
\downarrow
$$

$$
\texttt{gaussian\_clayton\_gas\_update}
\!\left(\mathbf f_{G,i},\mathbf s_{G,i},\boldsymbol\omega_G,\mathbf A_G,\mathbf B_G,f_{C,i},s_{C,i},\omega_C,A_C,B_C\right)
$$

$$
\downarrow
$$

$$
\mathbf f_{G,i+1}
=\boldsymbol\omega_G+\mathbf A_G\odot\mathbf s_{G,i}+\mathbf B_G\odot\mathbf f_{G,i},
\qquad
f_{C,i+1}=\omega_C+A_Cs_{C,i}+B_Cf_{C,i}
$$

$$
\downarrow
$$

$$
\texttt{gaussian\_clayton\_mixture\_log\_likelihood}
$$

$$
\downarrow
$$

$$
\mathcal L(\Theta)=\sum_{i=1}^{t}\log c_{\mathrm{mix},i}
$$

$$
\downarrow
$$

$$
\texttt{estimate\_gaussian\_clayton\_gas}
\!\left(
\widetilde{\mathbf U}_{\mathrm{train}}
\right)
$$

$$
\downarrow
$$

$$
\widehat{\Theta}=\arg\min_{\Theta}\operatorname{NLL}(\Theta)=\arg\min_{\Theta}\left[-\frac{1}{t}
\sum_{i=1}^{t}\log c_{\mathrm{mix},i}\right]
$$

$$
\Theta
=\left(
w,
\boldsymbol\omega_G,
\mathbf A_G,
\mathbf B_G,
\omega_C,
A_C,
B_C
\right)
$$

$$
\widehat{\Theta}=\left(\widehat w,\widehat{\boldsymbol\omega}_G,\widehat{\mathbf A}_G,\widehat{\mathbf B}_G,\widehat{\omega}_C,\widehat A_C,\widehat B_C\right)
$$

$$
\downarrow
$$

$$
\texttt{compute\_time\_varying\_copula\_paths}
\!\left(
\widetilde{\mathbf U}_{\mathrm{train}},
\widehat{\Theta}
\right)
$$

$$
\downarrow
$$

$$
\left\{\widehat w, \widehat{\mathbf R}_i,\widehat{\theta}_i,\log\widehat c_{\mathrm{mix},i}\!\left(\widetilde{\mathbf U}_i\right),\widehat c_{\mathrm{mix},i}\!\left(\widetilde{\mathbf U}_i\right)\right\}_{i=1}^{t},
$$

### Gaussian Copula

**correlation matrix (3 dim)**
$$
\mathbf R_{\mathrm{GAS},t}=\begin{pmatrix}1&\rho_{\mathrm{GAS},t}^{(1,2)}&\rho_{\mathrm{GAS},t}^{(1,3)}\\\rho_{\mathrm{GAS},t}^{(1,2)}&1&\rho_{\mathrm{GAS},t}^{(2,3)}\\\rho_{\mathrm{GAS},t}^{(1,3)}&\rho_{\mathrm{GAS},t}^{(2,3)}&1\end{pmatrix}.
$$

where

$$
\rho_{\mathrm{GAS},t}^{(j)}=\frac{1-\exp\left(-f_t^{(j)}\right)}{1+\exp\left(-f_t^{(j)}\right)},\qquad j=1,2
$$

$$
\rho_{\mathrm{GAS},t}^{(j)}=\tanh\left(\frac{f_t^{(j)}}{2}\right).
$$

$$
\rho_{\mathrm{GAS},t}^{(2,3)}=\rho_{\mathrm{GAS},t}^{(1,2)}\rho_{\mathrm{GAS},t}^{(1,3)}+\rho_{\mathrm{GAS},t}^{(2,3)\mid 1}\sqrt{\left(1-\left(\rho_{\mathrm{GAS},t}^{(1,2)}\right)^2\right)\left(1-\left(\rho_{\mathrm{GAS},t}^{(1,3)}\right)^2\right)}.
$$


In [28]:
def gaussian_parameter_3d(f_G_i):

    eps = 1e-6
    # values between -1 and 1
    rho_tanh = (1.0 - eps) * torch.tanh(f_G_i / 2.0)

    rho_12_i = rho_tanh[0]
    rho_13_i = rho_tanh[1]
    
    rho_23_given_1_i = rho_tanh[2]
    rho_23_i = rho_12_i * rho_13_i + rho_23_given_1_i * torch.sqrt((1.0 - rho_12_i**2) * (1.0 - rho_13_i**2))

    one = f_G_i.new_tensor(1.0)

    R_i = torch.stack([
        torch.stack([one, rho_12_i, rho_13_i]),
        torch.stack([rho_12_i, one, rho_23_i]),
        torch.stack([rho_13_i, rho_23_i, one]) ])

    return R_i

$$
\mathbf z_t=\begin{pmatrix}z^{(1)}_{t}\\\vdots\\z^{(d)}_{t}\end{pmatrix},\qquad z^{(i)}_{t}=\Phi^{-1}\left(\widetilde U^{(i)}_{t}\right),\qquad i=1,\ldots,d
$$
$$
c^{\mathrm{gaussian}}_{R}\left(\widetilde{\mathbf U}_t;\mathbf R_{\mathrm{GAS},t}\right)=\left|\mathbf R_{\mathrm{GAS},t}\right|^{-1/2}\exp\left\{-\frac{1}{2}\mathbf z_t^{\mathsf T}\left(\mathbf R_{\mathrm{GAS},t}^{-1}-\mathbf I_d\right)\mathbf z_t\right\}
$$
$$
\ell_t=\log  c^{\mathrm{gaussian}}_{R}\left(\widetilde{\mathbf U}_t;\mathbf R_{\mathrm{GAS},t}\right)  =-\frac{1}{2}\log\left|\mathbf R_{\mathrm{GAS},t}\right|-\frac{1}{2}\mathbf z_t^{\mathsf T}\left(\mathbf R_{\mathrm{GAS},t}^{-1}-\mathbf I_d\right)\mathbf z_t
$$

In [29]:
def gaussian_copula_log_density(z_i, R_i):
    
    eps = 1e-6
    identity = torch.eye(R_i.shape[0], dtype=R_i.dtype, device=R_i.device)

    # Cholesky decomposition of the correlation matrix
    L_i = torch.linalg.cholesky(R_i + eps * identity)

    log_det_R_i = 2.0 * torch.log(torch.diagonal(L_i)).sum()

    # R_i^{-1} * z_i 
    R_inv_z_i = torch.cholesky_solve(z_i.unsqueeze(1), L_i).squeeze(1)

    log_copula_density_i = -0.5 * (log_det_R_i + (torch.dot(z_i, R_inv_z_i) - torch.dot(z_i, z_i)))

    return log_copula_density_i

### Clayton Copula

$$
\theta_t=\exp(f_{C,t})>0
$$

In [ ]:
def clayton_parameter(f_C_i):
    theta_i = torch.exp(f_C_i)
    return theta_i

$$
\log c_{C,t}
=-\left(\frac{1}{\theta_t}+d\right)\log\left[\sum_{i=1}^{d}\left(\widetilde U_t^{(i)}\right)^{-\theta_t}-d+1\right]
+\sum_{i=1}^{d}\log\left[1+(i-1)\theta_t\right]
-(\theta_t+1)\sum_{i=1}^{d}\log \widetilde U_t^{(i)}.
$$

$$
\log c_{C,t}=\log(1+\theta_t)+\log(1+2\theta_t)
-(\theta_t+1)\sum_{i=1}^{3}\log \widetilde U_t^{(i)}
-\left(3+\frac{1}{\theta_t}\right)\log\left[\sum_{i=1}^{3}\left(\widetilde U_t^{(i)}\right)^{-\theta_t}-2\right]
$$

In [ ]:
def clayton_copula_log_density(u_tilde_i, theta_i):
  
    d = u_tilde_i.shape[0]
    
    clayton_sum_i = torch.sum(u_tilde_i ** (-theta_i)) - d + 1.0
    coefficient_index = torch.arange(1, d, dtype=theta_i.dtype, device=theta_i.device)

    log_coefficient_i = torch.sum(torch.log(1.0 + coefficient_index * theta_i))
    log_marginal_term_i = -(theta_i + 1.0)  * torch.sum(torch.log(u_tilde_i))
    log_generator_term_i = -(d + 1.0 / theta_i) * torch.log(clayton_sum_i)

    log_copula_density_i = log_coefficient_i + log_marginal_term_i + log_generator_term_i
    

    return log_copula_density_i

### Mixture 

**Mixture log density**

$$
c_{\mathrm{mix},t}=w c_{G,t}+(1-w)c_{C,t}
$$


In [30]:
def gaussian_clayton_mixture_log_density(log_c_G_i, log_c_C_i, weight):
   
    log_weight = torch.nn.functional.logsigmoid(weight)
    log_one_minus_weight = torch.nn.functional.logsigmoid(-weight)

    log_c_mix_i = torch.logsumexp(torch.stack([ log_weight + log_c_G_i, log_one_minus_weight + log_c_C_i]), dim=0)

    return log_c_mix_i

**Score**

$$
\boldsymbol{\nabla}_t=\nabla_{(\mathbf f_{G,t},\,f_{C,t})}\log c_{\mathrm{mix},t}
=
\begin{pmatrix}
\displaystyle \frac{\partial \log c_{\mathrm{mix},t}}{\partial \mathbf f_{G,t}}\\
\displaystyle \frac{\partial \log c_{\mathrm{mix},t}}{\partial f_{C,t}}
\end{pmatrix}
=
\begin{pmatrix}
\mathbf r_{G,t}\\
r_{C,t}
\end{pmatrix}
$$

In [ ]:
def gaussian_clayton_mixture_raw_score(log_c_mix_i, f_G_i, f_C_i):
    
    raw_score_G_i, raw_score_C_i = torch.autograd.grad(log_c_mix_i, (f_G_i, f_C_i), create_graph=True)
    
    return raw_score_G_i, raw_score_C_i

**Identity scaling**

$$
\mathbf S_{G,t}=\mathbf I, \qquad S_{C,t}=1
$$


In [23]:
def identity_scaling(raw_score_G_i, raw_score_C_i):
    
    scaled_score_G_i = raw_score_G_i
    scaled_score_C_i = raw_score_C_i
    
    return scaled_score_G_i, scaled_score_C_i

**inverse information scaling**

$$
\mathbf S_i =\mathcal I_{i\mid i-1}^{-1}
$$

$$
\mathbf s_t=\mathbf S_t\boldsymbol{\nabla}_t
$$

In [ ]:
def gaussian_clayton_mixture_score_scaling(raw_score_G_i, S_G_i, raw_score_C_i, S_C_i):
    
    scaled_score_G_i = S_G_i @ raw_score_G_i
    scaled_score_C_i = S_C_i * raw_score_C_i
   
    return scaled_score_G_i, scaled_score_C_i

**GAS update equation**

Gaussian
$$
\mathbf f_{G,t+1}=\boldsymbol{\omega}_G+\mathbf A_G\mathbf s_{G,t} + \mathbf B_G\mathbf f_{G,t}
$$

Clayton
$$
f_{C,t+1}=\omega_C+A_C s_{C,t}+B_C f_{C,t}.
$$

In [22]:
def gaussian_clayton_gas_update(f_G_i, scaled_score_G_i, omega_G, A_G, B_G, 
                                f_C_i, scaled_score_C_i, omega_C, A_C, B_C):
    
    f_G_next = omega_G + A_G * scaled_score_G_i + B_G * f_G_i
    f_C_next = omega_C + A_C * scaled_score_C_i + B_C * f_C_i
    
    return f_G_next, f_C_next

### Estimate

$$
\mathcal{L}=\sum_{t=1}^{T}\log\{c_{mix,i}\}
$$

$$
\mathcal{L}=\sum_{i=1}^{t}\log\{w\,c_G\left(u_i;R_i\right)+(1-w)\,c_C\left(u_i;\theta_i\right)\}
$$

In [25]:
def gaussian_clayton_mixture_log_likelihood(u_tilde, z, weight,
                                             f_G_0, omega_G, A_G, B_G,
                                             f_C_0, omega_C, A_C, B_C):

    f_G_i = f_G_0
    f_C_i = f_C_0
    log_likelihood = u_tilde.new_tensor(0.0)

    for i in range(u_tilde.shape[0]):
        R_i = gaussian_parameter_3d(f_G_i)
        theta_i = clayton_parameter(f_C_i)

        # copula log density
        log_c_G_i = gaussian_copula_log_density(z[i], R_i)
        log_c_C_i = clayton_copula_log_density(u_tilde[i], theta_i)

        # mixture
        log_c_mix_i = gaussian_clayton_mixture_log_density(log_c_G_i, log_c_C_i, weight)

        # score
        raw_score_G_i, raw_score_C_i = gaussian_clayton_mixture_raw_score(log_c_mix_i, f_G_i, f_C_i)
        scaled_score_G_i, scaled_score_C_i = identity_scaling(raw_score_G_i, raw_score_C_i)

        
        log_likelihood = log_likelihood + log_c_mix_i

        f_G_i, f_C_i = gaussian_clayton_gas_update(f_G_i, scaled_score_G_i, omega_G, A_G, B_G,
                                                   f_C_i, scaled_score_C_i, omega_C, A_C, B_C)

    return log_likelihood

$$
\hat{\Theta}=\arg\min_{\Theta}\operatorname{NLL}(\Theta)
$$

$$
\hat{\Theta}=\left(\widehat w,\widehat{\boldsymbol\omega}_G,\widehat{\mathbf A}_G,\widehat{\mathbf B}_G,\widehat{\omega}_C,\widehat A_C,\widehat B_C\right)
$$

In [1]:
def estimate_gaussian_clayton_gas(u_tilde, epochs=500, learning_rate=0.01):

    u_tilde = torch.as_tensor(u_tilde, dtype=torch.float64).clamp(1e-6, 1.0 - 1e-6)
    
    normal = torch.distributions.Normal(u_tilde.new_tensor(0.0), u_tilde.new_tensor(1.0))
    z = normal.icdf(u_tilde)

    # static parameters

    # mixture weight 
    weight = torch.nn.Parameter(u_tilde.new_tensor(0.0))

    # gaussian 
    omega_G = torch.nn.Parameter(u_tilde.new_zeros(3))
    A_G = torch.nn.Parameter(u_tilde.new_full((3,), 0.001))
    B_G = torch.nn.Parameter(u_tilde.new_full((3,), 0.90))

    # clayton 
    omega_C = torch.nn.Parameter(u_tilde.new_tensor(0.0))
    A_C = torch.nn.Parameter(u_tilde.new_tensor(0.001))
    B_C = torch.nn.Parameter(u_tilde.new_tensor(0.90))

    parameters = [weight,
                  omega_G, A_G, B_G,
                  omega_C, A_C, B_C]

    optimizer = torch.optim.Adam(parameters, lr=learning_rate)

    f_G_0 = u_tilde.new_zeros(3).requires_grad_(True)
    f_C_0 = u_tilde.new_tensor(0.0).requires_grad_(True)

    for _ in range(epochs):
        
        optimizer.zero_grad(set_to_none=True)
        
        nll =  - gaussian_clayton_mixture_log_likelihood(u_tilde, z, weight,
                                                          f_G_0, omega_G, A_G, B_G,
                                                          f_C_0, omega_C, A_C, B_C )
        mean_nll = nll / u_tilde.shape[0]
        
        mean_nll.backward()
        optimizer.step()

    return { "weight": weight.detach(),
             "omega_G": omega_G.detach(),
             "A_G": A_G.detach(),
             "B_G": B_G.detach(),
             "omega_C": omega_C.detach(),
             "A_C": A_C.detach(),
             "B_C": B_C.detach() }

### Compute time-varying mixture copula

In [31]:
def compute_time_varying_copula_paths(u_tilde, estimated_parameters):
   
    u_tilde = torch.as_tensor( u_tilde, dtype=torch.float64).clamp(1e-6, 1.0 - 1e-6)

    normal = torch.distributions.Normal(u_tilde.new_tensor(0.0), u_tilde.new_tensor(1.0))
    z = normal.icdf(u_tilde)

    # mixture weight
    weight = estimated_parameters["weight"].to( dtype=u_tilde.dtype, device=u_tilde.device)

    # gaussian
    omega_G = estimated_parameters["omega_G"].to(u_tilde)
    A_G = estimated_parameters["A_G"].to(u_tilde)
    B_G = estimated_parameters["B_G"].to(u_tilde)

    # clayton
    omega_C = estimated_parameters["omega_C"].to(u_tilde)
    A_C = estimated_parameters["A_C"].to(u_tilde)
    B_C = estimated_parameters["B_C"].to(u_tilde)

    f_G_i = u_tilde.new_zeros(3).requires_grad_(True)
    f_C_i = u_tilde.new_tensor(0.0).requires_grad_(True)

    R_path = []
    theta_path = []
    log_c_mix_path = []
    
    for i in range(u_tilde.shape[0]):
        
        R_i = gaussian_parameter_3d(f_G_i)
        theta_i = clayton_parameter(f_C_i)

        log_c_G_i = gaussian_copula_log_density(z[i], R_i)
        log_c_C_i = clayton_copula_log_density( u_tilde[i], theta_i)
        # log{ w * c_G + (1-w) * c_C }
        log_c_mix_i = gaussian_clayton_mixture_log_density(log_c_G_i, log_c_C_i, weight)
        
        score_G_i, score_C_i = torch.autograd.grad(log_c_mix_i, (f_G_i, f_C_i))
        scaled_score_G_i, scaled_score_C_i = identity_scaling(score_G_i,score_C_i)


        f_G_next, f_C_next = gaussian_clayton_gas_update(f_G_i, scaled_score_G_i, omega_G, A_G, B_G,
                                                   f_C_i, scaled_score_C_i, omega_C, A_C, B_C)

        f_G_i = f_G_next.detach().requires_grad_(True)
        f_C_i = f_C_next.detach().requires_grad_(True)

        R_path.append(R_i.detach())
        theta_path.append(theta_i.detach())
        log_c_mix_path.append(log_c_mix_i.detach())

    R_path = torch.stack(R_path)
    theta_path = torch.stack(theta_path)
    log_c_mix_path = torch.stack(log_c_mix_path)

    return {
        "weight": torch.sigmoid(weight.detach()),
        "R_path": R_path,
        "theta_path": theta_path,
        "log_c_mix_path": log_c_mix_path,
        "copula_density_path": torch.exp(log_c_mix_path)}